In [1]:
import os
from google.colab import userdata

# Groq
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# LangSmith tracing — these 3 lines enable automatic tracing
# Every LangGraph run will appear in your LangSmith dashboard
os.environ["LANGCHAIN_TRACING_V2"] = userdata.get("LANGCHAIN_TRACING_V2")
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "week2-crag-agent"  # project name in LangSmith

print("✓ LangSmith tracing enabled")
print(f"  Project: week2-crag-agent")
print(f"  View traces at: https://smith.langchain.com")

✓ LangSmith tracing enabled
  Project: week2-crag-agent
  View traces at: https://smith.langchain.com


In [2]:
!pip install -q langgraph langchain-openai chromadb sentence-transformers wikipedia-api fastapi uvicorn nest-asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 128.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import os
import warnings
warnings.filterwarnings("ignore")

from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("✓ Setup done")

✓ Setup done


In [4]:
import uuid
import chromadb
from chromadb.utils import embedding_functions

chroma_client = chromadb.Client()  # ephemeral in Colab
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
collection = chroma_client.get_or_create_collection(
    name="ai_knowledge",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:

CORPUS = [
    {
        "title": "RAG Overview",
        "source": "https://arxiv.org/abs/2005.11401",
        "content": """Retrieval-Augmented Generation (RAG) combines a retrieval system with a language model. The model retrieves relevant documents from an external knowledge base and uses them as context when generating responses. RAG addresses a key limitation of parametric language models: their knowledge is fixed at training time. By retrieving from an external store, the model can access up-to-date information without retraining. The RAG pipeline has three stages: indexing (chunking and embedding documents), retrieval (finding similar chunks), and generation (passing chunks as context to the LLM). RAG significantly reduces hallucination because responses are grounded in retrieved documents rather than parametric memory."""
    },
    {
        "title": "CRAG Paper",
        "source": "https://arxiv.org/abs/2401.15884",
        "content": """Corrective Retrieval Augmented Generation (CRAG) addresses a weakness of standard RAG: it blindly uses whatever was retrieved, even if irrelevant. CRAG adds a relevance grading step after retrieval. An evaluator scores each retrieved document as relevant or not relevant. Based on these scores, CRAG decides whether to use the retrieved documents or fall back to web search. The corrective mechanism prevents the generator from hallucinating based on irrelevant context. CRAG outperforms standard RAG on knowledge-intensive tasks, particularly when the query involves specialized knowledge not well-covered in the retrieval corpus. CRAG can be implemented as a LangGraph pipeline: retrieve node, grade_relevance node with conditional routing, optional web_search node, and generate node."""
    },
    {
        "title": "Vector Databases",
        "source": "https://www.pinecone.io/learn/vector-database/",
        "content": """Vector databases store high-dimensional embeddings and enable fast similarity search. Unlike traditional databases that match exact values, vector databases find semantically similar content using distance metrics like cosine similarity and dot product. Popular vector databases include ChromaDB (open-source, local-first, great for development), Pinecone (managed cloud service, production-ready, auto-scaling), Qdrant (open-source, supports filtering, self-hostable), and Weaviate (open-source, multi-modal support). HNSW (Hierarchical Navigable Small World) is the most common indexing algorithm for approximate nearest neighbor search. For production RAG, Qdrant and Pinecone are the most commonly used options."""
    },
    {
        "title": "LangGraph Framework",
        "source": "https://langchain-ai.github.io/langgraph/",
        "content": """LangGraph is a framework for building stateful, multi-actor LLM applications. It supports cyclic graphs which enables agent loops. The core abstraction is the StateGraph: nodes are Python functions that read from and write to a shared TypedDict state. Edges define control flow, either unconditional or conditional based on current state. Key production features include checkpointing (saves state after every node), human-in-the-loop (interrupt_before pauses execution), streaming (values, updates, messages modes), and thread isolation via thread_id. LangGraph is the recommended framework for production AI agents because it provides observability, persistence, and control flow features that simple ReAct loops lack."""
    },
    {
        "title": "Chunking Strategies",
        "source": "https://www.llamaindex.ai/blog/chunking-strategies",
        "content": """Chunking strategy significantly affects RAG quality. Fixed-size chunking splits text into equal-length pieces regardless of content boundaries. Simple but often cuts sentences mid-thought. Recursive character chunking tries to split at natural boundaries: paragraphs first, then sentences, then words. Add 10-20% overlap to preserve context across boundaries. Semantic chunking groups sentences by meaning using embedding similarity. More expensive but produces more coherent chunks. Parent-document retrieval stores small chunks for precise retrieval but returns larger parent chunks for generation. Chunk size of 256-512 tokens works well for most RAG applications."""
    },
    {
        "title": "RAGAS Evaluation",
        "source": "https://docs.ragas.io/",
        "content": """RAGAS evaluates RAG systems without requiring human-labeled ground truth. Four key metrics: Faithfulness measures whether claims in the answer are supported by retrieved context (score 0-1). Answer Relevance measures whether the answer addresses the question. Context Precision measures what fraction of retrieved chunks were useful. Context Recall measures whether retrieval found all necessary information. Faithfulness is the most important metric for production RAG because it directly measures hallucination. A faithfulness score below 0.7 indicates the model is adding information not present in retrieved context. To evaluate faithfulness: decompose the answer into claims, check each claim against context using an LLM judge, compute the fraction of supported claims."""
    },
    {
        "title": "Embedding Models",
        "source": "https://huggingface.co/sentence-transformers",
        "content": """Embedding models convert text into dense vector representations capturing semantic meaning. all-MiniLM-L6-v2 is the most popular lightweight model: 384 dimensions, runs locally on CPU, good quality for English text, completely free. Ideal for development and cost-sensitive deployments. OpenAI text-embedding-3-small produces 1536-dimensional embeddings with higher quality, especially for multilingual content. BGE-M3 from BAAI supports 100+ languages, multiple retrieval modes (dense, sparse, multi-vector), and long documents up to 8192 tokens. For production RAG on a budget, all-MiniLM-L6-v2 is the standard starting point."""
    },
    {
        "title": "Hallucination Reduction",
        "source": "https://arxiv.org/abs/2309.01219",
        "content": """Hallucination in RAG occurs when the generated answer contains information not present in retrieved context. Key techniques: citation grounding (require model to cite specific passages), NLI verification (check each claim is entailed by context), faithfulness scoring with RAGAS, temperature 0 (reduces creative deviation), and explicit system prompt instruction to only use provided context. CRAG reduces hallucination at the retrieval stage by ensuring only relevant context reaches the generator. The combination of CRAG filtering and explicit prompting is the most effective approach for production RAG systems."""
    },
]

In [6]:
!pip install -q langchain-text-splitters

In [7]:
# Replace chunk_text and ingest_documents with this:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    # splits at these boundaries in order: paragraph, sentence, word
    separators=["\n\n", "\n", ". ", " ", ""],
)

def ingest_documents():
    all_chunks, all_metadatas, all_ids = [], [], []

    for doc in CORPUS:
        # LangChain handles boundary-aware splitting
        chunks = text_splitter.split_text(doc["content"])

        for i, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            all_metadatas.append({
                "title": doc["title"],
                "source": doc["source"],
                "chunk_index": i,
                "total_chunks": len(chunks),
            })
            all_ids.append(str(uuid.uuid4()))

    collection.add(
        documents=all_chunks,
        metadatas=all_metadatas,
        ids=all_ids,
    )
    print(f"✓ Ingested {len(CORPUS)} docs → {len(all_chunks)} chunks")
    print(f"✓ Collection size: {collection.count()} chunks")

In [8]:
ingest_documents()

✓ Ingested 8 docs → 16 chunks
✓ Collection size: 16 chunks


In [9]:
import time
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    temperature=0,
    max_tokens=1000,
)

In [10]:
class RAGState(TypedDict):
    question: str
    documents: list[str]
    sources: list[str]
    grades: list[str]
    generation: str
    web_search_needed: bool


In [11]:
def retrieve(state: RAGState) -> dict:
    results = collection.query(
        query_texts=[state["question"]],
        n_results=min(4, collection.count()),
    )
    documents = results["documents"][0]
    sources = [m["source"] for m in results["metadatas"][0]]
    print(f"📚 Retrieved {len(documents)} chunks")
    return {"documents": documents, "sources": sources, "grades": [], "web_search_needed": False}

In [12]:

def grade_relevance(state: RAGState) -> dict:
    grades = []
    for doc in state["documents"]:
        response = llm.invoke([
            SystemMessage(content="You are a relevance grader. Reply only with 'relevant' or 'not_relevant'."),
            HumanMessage(content=(
                f"Question: {state['question']}\n\n"
                f"Document: {doc[:400]}\n\n"
                f"Is this document useful for answering the question? "
                f"Reply with one word: relevant or not_relevant"
            ))
        ])
        grade = "not_relevant" if "not" in response.content.lower() else "relevant"
        grades.append(grade)
        time.sleep(1)

    relevant_count = grades.count("relevant")
    web_search_needed = relevant_count < len(grades) / 2
    print(f"🔍 Grades: {grades} | Web search needed: {web_search_needed}")
    return {"grades": grades, "web_search_needed": web_search_needed}

In [13]:
def web_search(state: RAGState) -> dict:
    print("🌐 Web search fallback")
    try:
        import wikipediaapi
        wiki = wikipediaapi.Wikipedia(language='en', user_agent='CRAG/1.0')
        page = wiki.page(state["question"].replace("?", "").strip())
        if page.exists():
            content = f"[Web]\nTitle: {page.title}\n{page.summary[:600]}"
        else:
            content = f"No web results for '{state['question']}'"
    except Exception as e:
        content = f"Web search error: {e}"
    return {
        "documents": state["documents"] + [content],
        "sources": state["sources"] + ["Web Search"],
    }


In [14]:
def generate(state: RAGState) -> dict:
    if state.get("grades"):
        context_docs = [
            doc for doc, grade in zip(state["documents"][:len(state["grades"])],
                                     state["grades"]) if grade == "relevant"
        ]
        context_docs.extend(state["documents"][len(state["grades"]):])
    else:
        context_docs = state["documents"]

    if not context_docs:
        context_docs = state["documents"]

    context = "\n\n---\n\n".join(context_docs[:4])
    response = llm.invoke([
        SystemMessage(content=(
            "Answer using ONLY the provided context. "
            "Cite which source you used. "
            "If context is insufficient, say so."
        )),
        HumanMessage(content=f"Question: {state['question']}\n\nContext:\n{context}\n\nAnswer:")
    ])
    print("✍️  Generated answer")
    return {"generation": response.content}

In [15]:
def route_after_grading(state:RAGState)->str:
  return 'web_search' if state['web_search_needed'] else 'generate'

In [16]:
def build_crag_graph():
    graph = StateGraph(RAGState)
    graph.add_node("retrieve", retrieve)
    graph.add_node("grade_relevance", grade_relevance)
    graph.add_node("web_search", web_search)
    graph.add_node("generate", generate)
    graph.add_edge(START, "retrieve")
    graph.add_edge("retrieve", "grade_relevance")
    graph.add_conditional_edges(
        "grade_relevance",
        route_after_grading,
        {"generate": "generate", "web_search": "web_search"}
    )
    graph.add_edge("web_search", "generate")
    graph.add_edge("generate", END)
    return graph.compile()

In [17]:
def ask(question: str) -> dict:
    graph = build_crag_graph()
    result = graph.invoke({"question": question})
    return {
        "question": question,
        "answer": result["generation"],
        "web_search_used": result["web_search_needed"],
        "relevant_chunks": result["grades"].count("relevant"),
        "total_chunks": len(result["grades"]),
    }

print("✓ CRAG pipeline ready")

✓ CRAG pipeline ready


In [18]:
# Test 1 — in corpus, should NOT need web search
r1 = ask("What is CRAG and how does it improve basic RAG?")
print(f"\nAnswer: {r1['answer'][:300]}")
print(f"Web search: {r1['web_search_used']} | Relevant: {r1['relevant_chunks']}/{r1['total_chunks']}")

time.sleep(5)

# Test 2 — in corpus
r2 = ask("Which chunking strategy is best for RAG?")
print(f"\nAnswer: {r2['answer'][:300]}")
print(f"Web search: {r2['web_search_used']} | Relevant: {r2['relevant_chunks']}/{r2['total_chunks']}")

time.sleep(5)

# Test 3 — NOT in corpus, should trigger web search
r3 = ask("Who is the president of Egypt?")
print(f"\nAnswer: {r3['answer'][:300]}")
print(f"Web search: {r3['web_search_used']} | Relevant: {r3['relevant_chunks']}/{r3['total_chunks']}")

📚 Retrieved 4 chunks
🔍 Grades: ['not_relevant', 'relevant', 'relevant', 'not_relevant'] | Web search needed: False
✍️  Generated answer

Answer: **CRAG (Corrective Retrieval Augmented Generation)** is a refinement of standard Retrieval‑Augmented Generation (RAG) that tackles the problem of hallucination caused by irrelevant retrieved content.  
- It introduces a *relevance‑grading* step immediately after retrieval.  
- An evaluator scores ea
Web search: False | Relevant: 2/4
📚 Retrieved 4 chunks
🔍 Grades: ['relevant', 'not_relevant', 'not_relevant', 'relevant'] | Web search needed: False
✍️  Generated answer

Answer: The most effective strategy for RAG is **semantic chunking**.  
It groups sentences by meaning using embedding similarity, which yields the most coherent chunks—though it is more computationally expensive.  
(“Semantic chunking groups sentences by meaning using embedding similarity. More expensive b
Web search: False | Relevant: 2/4
📚 Retrieved 4 chunks
🔍 Grades: ['not_rel

In [19]:
import json

def evaluate_faithfulness(question: str, answer: str, context_docs: list[str]) -> dict:
  context = '\n\n'.join(context_docs[:3])
  decompose = llm.invoke([
        SystemMessage(content="Extract factual claims from the answer. Return ONLY a JSON array of strings."),
        HumanMessage(content=f"Answer: {answer}\n\nReturn JSON array: [\"claim1\", \"claim2\", ...]")
    ])
  try:
      raw = decompose.content.strip().replace("```json","").replace("```","").strip()
      claims = json.loads(raw)
      if not isinstance(claims, list):
        claims = [answer[:200]]
  except Exception:
    claims = [answer[:200]]
  supported = 0
  details = []
  for claim in claims[:4]:
    verify = llm.invoke([
        SystemMessage(content="Check if claim is supported by context. Reply: supported or not_supported"),
        HumanMessage(content=f"Context:\n{context[:600]}\n\nClaim: {claim}\n\nReply: supported or not_supported")
    ])
    verdict = 'not_supported' if 'not' in verify.content.lower() else 'supported'
    if verdict == 'supported':
      supported += 1
    details.append({'claim': claim[:60], 'verdict': verdict})
    time.sleep(1)
  total = len(claims[:4])
  score = round(supported / total, 2) if total > 0 else 0.0
  return {'score': score, 'supported': supported, 'total': total, 'details': details}

In [20]:
TEST_CASES = [
    "What is RAG and how does it work?",
    "How does CRAG improve basic RAG?",
    "Which vector database should I use for production?",
    "What is the best chunking strategy for RAG?",
]

print("═"*60)
print("  EVALUATION SUITE")
print("═"*60)

════════════════════════════════════════════════════════════
  EVALUATION SUITE
════════════════════════════════════════════════════════════


In [21]:
scores = []
for question in TEST_CASES:
    print(f"\nEvaluating: {question[:50]}...")
    result = ask(question)
    docs = collection.query(query_texts=[question], n_results=3)["documents"][0]
    eval_result = evaluate_faithfulness(question, result["answer"], docs)
    scores.append(eval_result["score"])
    print(f"  Faithfulness: {eval_result['score']} | Web search: {result['web_search_used']}")
    time.sleep(5)

avg = round(sum(scores) / len(scores), 2)
print(f"\n{'─'*40}")
print(f"Average faithfulness: {avg}")
print(f"Target: > 0.70")
print(f"Status: {'✓ PASS' if avg > 0.70 else '✗ FAIL - needs improvement'}")


Evaluating: What is RAG and how does it work?...
📚 Retrieved 4 chunks
🔍 Grades: ['not_relevant', 'relevant', 'not_relevant', 'not_relevant'] | Web search needed: True
🌐 Web search fallback
✍️  Generated answer
  Faithfulness: 1.0 | Web search: True

Evaluating: How does CRAG improve basic RAG?...
📚 Retrieved 4 chunks
🔍 Grades: ['relevant', 'relevant', 'not_relevant', 'not_relevant'] | Web search needed: False
✍️  Generated answer
  Faithfulness: 1.0 | Web search: False

Evaluating: Which vector database should I use for production?...
📚 Retrieved 4 chunks
🔍 Grades: ['relevant', 'not_relevant', 'not_relevant', 'not_relevant'] | Web search needed: True
🌐 Web search fallback
✍️  Generated answer
  Faithfulness: 1.0 | Web search: True

Evaluating: What is the best chunking strategy for RAG?...
📚 Retrieved 4 chunks
🔍 Grades: ['not_relevant', 'relevant', 'not_relevant', 'relevant'] | Web search needed: False
✍️  Generated answer
  Faithfulness: 0.75 | Web search: False

────────────────────

In [22]:
import nest_asyncio
import threading
import uvicorn
import requests
from fastapi import FastAPI

nest_asyncio.apply()

app = FastAPI(title="CRAG API")

@app.get("/ask")
def ask_endpoint(question: str):
    result = ask(question)
    return result

@app.get("/health")
def health():
    return {"status": "ok", "chunks": collection.count()}

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

# Start in background thread
thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)
print("✓ Server running on port 8000")

# Test
response = requests.get(
    "http://localhost:8000/ask",
    params={"question": "What is RAG?"}
)
print(f"\nAPI response: {response.json()['answer'][:200]}")

✓ Server running on port 8000
📚 Retrieved 4 chunks
🔍 Grades: ['not_relevant', 'relevant', 'not_relevant', 'not_relevant'] | Web search needed: True
🌐 Web search fallback
✍️  Generated answer

API response: RAG (Retrieval‑Augmented Generation) is a pipeline that combines document indexing, retrieval, and LLM generation. It first indexes documents by chunking and embedding them, then retrieves the most si
